# 05 — Storytelling Executivo

Objetivos:
- Consolidar achados em visualizações executivas
- Produzir uma tabela final: cluster → perfil → risco/valor → KPI esperado


In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from src.features import build_modeling_dataframe
from src.modeling import load_pipeline, transform_for_model
from src.viz import pca_2d_plot, radar_chart, set_style

set_style()

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'base_cliente_clusterizada.parquet')
pipe = load_pipeline(str(REPORTS_DIR / 'kmeans_pipeline.joblib'))
X, _ = build_modeling_dataframe(df)
Xt = transform_for_model(pipe, X)
df.shape


## 1) PCA 2D com clusters


In [ ]:
labels = df['cluster_id'].astype(int).values
ax = pca_2d_plot(Xt, labels, title='PCA 2D — visão cliente-atual')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'pca_2d_clusters.png', dpi=150, bbox_inches='tight')
plt.show()


## 2) Heatmap de perfis (médias normalizadas)


In [ ]:
key = [c for c in ['log_acessos','dias_sem_acessar','recorrente','parcelado','nunca_acessou','n_transacoes_cliente','recencia_compra_dias'] if c in df.columns]
prof = df.groupby(['cluster_id','nome_cluster'])[key].mean().reset_index()

mat = prof.set_index(['cluster_id','nome_cluster'])[key]
mat_norm = (mat - mat.min()) / (mat.max() - mat.min()).replace(0, 1)

plt.figure(figsize=(12, 6))
sns.heatmap(mat_norm, cmap='YlGnBu', linewidths=0.2)
plt.title('Perfis normalizados por cluster (média)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'heatmap_perfis.png', dpi=150, bbox_inches='tight')
plt.show()


## 3) Radar chart (Plotly)


In [ ]:
rad = mat_norm.reset_index()
rad['cluster_id'] = rad['cluster_id'].astype(int)
fig = radar_chart(rad.drop(columns=['nome_cluster']), cluster_col='cluster_id')
fig.write_html(str(REPORTS_DIR / 'radar_clusters.html'))
fig


## 4) Tabela executiva final


In [ ]:
def perfil_curto(row):
    return (
        f"log_acessos={row.get('log_acessos', np.nan):.2f} | "
        f"inatividade={row.get('dias_sem_acessar', np.nan):.0f}d | "
        f"%recorrente={row.get('recorrente', np.nan):.2f} | "
        f"%nunca_acessou={row.get('nunca_acessou', np.nan):.2f}"
    )

summary = df.groupby(['cluster_id','nome_cluster']).agg(
    tamanho=('cliente','size'),
    log_acessos=('log_acessos','mean'),
    dias_sem_acessar=('dias_sem_acessar','mean'),
    recorrente=('recorrente','mean'),
    parcelado=('parcelado','mean'),
    nunca_acessou=('nunca_acessou','mean'),
    renovacao=('renovacao', lambda s: (s=='sim').mean() if s.dtype=='object' else np.nan),
).reset_index()

summary['perfil'] = summary.apply(perfil_curto, axis=1)
summary['risco_valor'] = (
    summary['log_acessos'].rank(pct=True)
    + summary['recorrente'].rank(pct=True)
    - summary['dias_sem_acessar'].rank(pct=True)
)
summary['kpi_esperado'] = 'Ativação/engajamento ↑ | churn ↓ | renovação ↑ | NPS ↑'
summary = summary.sort_values('risco_valor', ascending=False)

summary.to_excel(REPORTS_DIR / 'tabela_executiva.xlsx', index=False)
summary
